# TSL Translator - Data Exploration & Demo

This notebook demonstrates the TSL Translator pipeline:
1. Data exploration
2. Model training
3. Real-time translation demo

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Import TSL modules
from src.languages.dictionary import TSLDictionary
from src.models.landmark_extractor import MediaPipeLandmarkExtractor
from src.translation.translator import TSLTranslator

## 1. TSL Dictionary Exploration

In [ ]:
# Initialize dictionary
dictionary = TSLDictionary()

print(f"Total TSL signs: {len(dictionary)}")
print(f"\nAll TSL labels: {dictionary.get_all_labels()}")
print(f"\nAll Tunisian words: {dictionary.get_all_tunisian_words()}")

In [ ]:
# Example translations
test_labels = ['labes', 'mch', '7aj', 'tunis', 'lbed']

print("Example Translations:")
print("=" * 50)
for label in test_labels:
    result = dictionary.get_all_translations(label)
    print(f"\nTSL: {label}")
    print(f"  Tunisian: {result['tunisian']}")
    print(f"  French: {result['french']}")
    print(f"  English: {result['english']}")
    print(f"  Arabic: {result['arabic']}")

## 2. Sign Distribution

In [ ]:
# Categories distribution
categories = {
    'Demandes': 24,
    'Destinations': 8,
    'Famille': 12,
    'Jours': 7,
    'Transport': 6
}

plt.figure(figsize=(10, 6))
plt.bar(categories.keys(), categories.values(), color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
plt.title('TSL Signs Distribution by Category')
plt.xlabel('Category')
plt.ylabel('Number of Signs')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. MediaPipe Landmark Extraction Demo

In [ ]:
# Initialize landmark extractor
extractor = MediaPipeLandmarkExtractor()

# Create a test image (white background with a simple hand-like shape)
test_image = np.ones((224, 224, 3), dtype=np.uint8) * 255

# Test extraction
landmarks = extractor.extract_landmarks(test_image)
print(f"Landmarks shape: {landmarks.shape if landmarks is not None else None}")
print(f"Features shape: {extractor.extract_features(test_image).shape if extractor.extract_features(test_image) is not None else None}")

## 4. Model Architecture

In [ ]:
from src.models.classifier import TSLClassifier
import torch

# Create model
model = TSLClassifier(num_classes=57)

# Print model summary
print("Model Architecture:")
print(model)

# Test forward pass
dummy_input = torch.randn(1, 126)
output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Translation Demo

In [ ]:
# Test text translation
from src.translation.translator import translate_tsl_to_text, translate_text_to_tsl

# TSL to Text
print("TSL to Text Translation:")
print("=" * 40)
test_tsl = ['labes', 'mch', '7aj']
for tsl in test_tsl:
    result = translate_tsl_to_text(tsl)
    print(f"{tsl} -> {result['tunisian']} ({result['french']})")

print("\n")

# Text to TSL
print("Text to TSL Translation:")
print("=" * 40)
test_text = ['bien', 'non', 'père']
for text in test_text:
    result = translate_text_to_tsl(text)
    print(f"{text} -> {result}")

## 6. Create Demo Video

In [ ]:
# Create a demo visualization
def create_demo_frame(tsl_label, tunisian, french, english):
    """Create a demo frame with translations"""
    frame = np.ones((300, 400, 3), dtype=np.uint8) * 40  # Dark background
    
    # Add text
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(frame, f"TSL: {tsl_label}", (20, 50), font, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Tunisian: {tunisian}", (20, 100), font, 0.8, (255, 255, 0), 2)
    cv2.putText(frame, f"French: {french}", (20, 150), font, 0.8, (255, 0, 0), 2)
    cv2.putText(frame, f"English: {english}", (20, 200), font, 0.8, (0, 255, 255), 2)
    
    return frame

# Create demo frames
demo_signs = [
    ('labes', 'bien', 'well', 'bien'),
    ('mch', 'non', 'no', 'non'),
    ('7aj', 'père', 'father', 'père'),
    ('tunis', 'Tunis', 'Tunis', 'Tunis'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (tsl, tun, fr, en) in zip(axes, demo_signs):
    frame = create_demo_frame(tsl, tun, fr, en)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"TSL: {tsl}")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. TSL dictionary with 57 signs across 5 categories
2. MediaPipe landmark extraction
3. Model architecture (MLP with 126 input features)
4. Translation pipeline (TSL → Tunisian → French/English/Arabic)

### Next Steps
- Train model on actual TSL dataset
- Add more signs
- Implement real-time webcam translation